## 연습 1: AutoSphere 데이터 플랫폼 설정

당신은 글로벌 자동차 제조업체인 **AutoSphere AG**의 데이터 엔지니어입니다. 당신의 커넥티드 차량 플랫폼은 수백만 대의 차량에서 텔레메트리를 수집하고, 고객 등록 데이터를 관리하며, 서비스 이력을 추적합니다.

거버넌스 제어를 적용하려면 먼저 플랫폼의 기반이 되는 카탈로그, 스키마 및 테이블을 만들어야 합니다. 또한 각 테이블과 열의 목적을 문서화하기 위해 **주석**을 추가하여 — 데이터 자산을 다른 팀이 검색할 수 있도록 만듭니다.

### 설정 — 카탈로그, 스키마 및 테이블 만들기

거버넌스 제어에 집중할 수 있도록 AutoSphere 데이터 플랫폼이 아래 셀에서 설정됩니다. 다음을 만듭니다:

- **`automotive_catalog`** — 설명 주석이 포함된 AutoSphere 플랫폼용 Unity Catalog 카탈로그
- **`automotive_catalog.governance_lab`** — 이 랩 전반에서 사용되는 스키마
- **`customer_registrations`** — 등록된 고객 및 연결된 차량당 한 행; 테이블 수준 주석 포함
- **`vehicle_telemetry`** — 커넥티드 차량의 연속적인 텔레메트리 이벤트; `speed_kmh` 및 `battery_level_pct`에 대한 열 수준 주석 포함

셀을 실행하여 환경을 준비한 다음 연습 2로 계속 진행하세요.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS automotive_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'AutoSphere AG의 커넥티드 차량 플랫폼 카탈로그 — 등록, 텔레메트리 및 서비스 데이터를 저장합니다.'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS automotive_catalog.governance_lab
  COMMENT '실습용 Unity Catalog 연습을 위한 거버넌스 랩 스키마.';

-- customer_registrations 테이블 만들기
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.customer_registrations
  COMMENT '등록된 AutoSphere 고객 및 연결된 차량당 한 행.'
AS SELECT * FROM VALUES
  (1, 'Lukas Bauer',    'lukas.bauer@autosphere.de',   'DE-LB-4821', 'DE', 'VH-001'),
  (2, 'Sophie Martin',  'sophie.martin@autosphere.fr',  'FR-SM-9034', 'FR', 'VH-002'),
  (3, 'James Clarke',   'james.clarke@autosphere.uk',   'UK-JC-1172', 'UK', 'VH-003'),
  (4, 'Yuki Tanaka',    'yuki.tanaka@autosphere.jp',    'JP-YT-5561', 'JP', 'VH-004'),
  (5, 'Maria Santos',   'maria.santos@autosphere.pt',   'PT-MS-8843', 'PT', 'VH-005'),
  (6, 'Carlos Rivera',  'carlos.rivera@autosphere.es',  'ES-CR-3309', 'ES', 'VH-006')
AS t(customer_id, full_name, email, driver_license_no, country, vehicle_id);

-- vehicle_telemetry 테이블 만들기
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.vehicle_telemetry (
  event_id          BIGINT    COMMENT '고유 텔레메트리 이벤트 식별자',
  vehicle_id        STRING,
  event_time        TIMESTAMP,
  speed_kmh         INT       COMMENT '시간당 킬로미터 단위의 순간 차량 속도',
  battery_level_pct INT       COMMENT '백분율(0–100)로 나타낸 충전 상태',
  latitude          DOUBLE,
  longitude         DOUBLE,
  country           STRING
);

INSERT INTO automotive_catalog.governance_lab.vehicle_telemetry VALUES
  (1, 'VH-001', TIMESTAMP '2026-03-01 08:15:00', 112, 78,  48.8566,   2.3522, 'DE'),
  (2, 'VH-002', TIMESTAMP '2026-03-01 08:16:00', 95,  55,  51.5074,  -0.1278, 'FR'),
  (3, 'VH-003', TIMESTAMP '2026-03-01 08:17:00', 130, 91,  40.4168,  -3.7038, 'UK'),
  (4, 'VH-004', TIMESTAMP '2026-03-01 08:18:00', 88,  34,  35.6762, 139.6503, 'JP'),
  (5, 'VH-005', TIMESTAMP '2026-03-01 08:19:00', 105, 62,  38.7223,  -9.1393, 'PT'),
  (6, 'VH-006', TIMESTAMP '2026-03-01 08:20:00', 77,  47,  41.3851,   2.1734, 'ES');

## 연습 2: 거버넌스를 위한 데이터 자산 태그 지정

AutoSphere의 규정 준수 팀은 **개인 식별 정보(PII)**를 포함하는 모든 테이블과 열에 일관된 레이블을 지정하도록 요구합니다. 태그는 자동화된 검색을 가능하게 하며 나중에 액세스 정책을 구동합니다.

이 연습에서는 `customer_registrations` 테이블을 분류하기 위해 테이블 수준 및 열 수준 태그를 적용합니다.

### 작업 2.1 — 테이블 수준 태그 추가

`customer_registrations` 테이블에 태그를 지정하여 도메인과 민감도별로 분류하세요:

- `domain` = `customer`
- `data_classification` = `confidential`

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks SQL에서 Unity Catalog 테이블에 키-값 태그를 어떻게 추가하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... SET TAGS ('key' = 'value', 'key2' = 'value2')`.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.customer_registrations
SET TAGS ('domain' = 'customer', 'data_classification' = 'confidential');

### 작업 2.2 — 열 수준 PII 태그 추가

`customer_registrations`의 PII 열에 태그를 지정하여 민감한 정보로 식별하세요:

- `email` 열 → `pii` = `email`
- `driver_license_no` 열 → `pii` = `driver_license`

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks에서 SQL을 사용하여 Unity Catalog 테이블의 특정 열에 태그를 어떻게 추가하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... ALTER COLUMN <col> SET TAGS (...)`.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.customer_registrations
ALTER COLUMN email
SET TAGS ('pii' = 'email');

ALTER TABLE automotive_catalog.governance_lab.customer_registrations
ALTER COLUMN driver_license_no
SET TAGS ('pii' = 'driver_license');

### 작업 2.3 — 태그 확인

`system.information_schema.table_tags`를 사용하여 테이블 수준 태그가 적용되었는지 확인하세요. 그런 다음 열을 describe하여 열 태그가 있는지 확인하세요. 이를 위해 `system.information_schema.column_tags`를 사용하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks SQL에서 Unity Catalog 테이블과 그 열의 태그를 어떻게 확인할 수 있나요?"*

In [ ]:
%sql
-- customer_registrations의 테이블 수준 태그 표시

SELECT
  catalog_name,
  schema_name,
  table_name,
  tag_name,
  tag_value
FROM system.information_schema.table_tags
WHERE catalog_name = 'automotive_catalog'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations';

In [ ]:
%sql
-- customer_registrations의 열 수준 태그 표시

SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'automotive_catalog'
  AND schema_name = 'governance_lab'
  AND table_name = 'customer_registrations'
ORDER BY column_name, tag_name;

## 연습 3: 데이터 보존 구성

AutoSphere의 차량 텔레메트리 테이블은 지속적으로 기록됩니다. GDPR 데이터 최소화 원칙을 준수하고 스토리지 비용을 제어하려면 플랫폼은 다음을 수행해야 합니다:

- 테이블 기록을 **14일**만 보존합니다.
- `VACUUM`을 사용하여 오래된 데이터 파일을 영구적으로 제거합니다.
- 향후 유지 관리가 자동으로 실행되도록 **예측 최적화**를 활성화합니다.

이 연습에서는 보존 설정을 구성하고, 레코드 삭제를 시뮬레이션하며, `VACUUM` 작업을 실행합니다.

### 작업 3.1 — 보존 속성 구성

`vehicle_telemetry`에 다음 Delta Lake 테이블 속성을 설정하세요:

- `delta.logRetentionDuration` = `interval 14 days`
- `delta.deletedFileRetentionDuration` = `interval 14 days`

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks에서 Unity Catalog 테이블에 Delta Lake 보존 기간 속성을 어떻게 설정하나요?"*

**힌트:** 다음을 사용하세요: `ALTER TABLE ... SET TBLPROPERTIES (...)`.

In [ ]:
%sql
ALTER TABLE automotive_catalog.governance_lab.vehicle_telemetry
SET TBLPROPERTIES (
  'delta.logRetentionDuration'         = 'interval 14 days',
  'delta.deletedFileRetentionDuration' = 'interval 14 days'
);

### 작업 3.2 — GDPR 삭제 요청 시뮬레이션

고객이 삭제 요청권을 제출했습니다. `vehicle_telemetry` 테이블에서 차량 `VH-003`에 대한 모든 텔레메트리 레코드를 삭제하세요.

삭제 후 테이블을 쿼리하여 레코드가 사라졌는지 확인하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks SQL에서 Delta Lake 테이블의 특정 행을 어떻게 삭제하나요?"*

In [ ]:
%sql
DELETE FROM automotive_catalog.governance_lab.vehicle_telemetry
WHERE vehicle_id = 'VH-003';

-- 삭제 확인
SELECT * FROM automotive_catalog.governance_lab.vehicle_telemetry
ORDER BY event_id;

### 작업 3.3 — VACUUM을 실행하여 삭제된 데이터 파일 제거

삭제된 행은 `VACUUM`이 실행될 때까지 기본 Parquet 파일에 여전히 존재합니다. `vehicle_telemetry`에 `VACUUM`을 실행하여 현재 또는 최근 테이블 버전에서 더 이상 참조되지 않는 파일을 영구적으로 제거하세요.

기본 보존 기간인 **168시간(7일)**을 사용하세요. 이는 안전하고 서버리스 컴퓨팅에서 작동하며 랩 환경에 적합합니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks에서 Delta Lake 테이블에 VACUUM을 어떻게 실행하며 RETAIN 옵션은 어떤 역할을 하나요?"*

**참고:** 서버리스 컴퓨팅에서는 최소 보존 기간 재정의가 지원되지 않습니다. 항상 최소 168시간의 `RETAIN` 값을 지정하거나, 기본값을 사용하려면 `RETAIN`을 완전히 생략하세요.

In [ ]:
%sql
VACUUM automotive_catalog.governance_lab.vehicle_telemetry RETAIN 168 HOURS;

### 작업 3.4 — 예측 최적화 활성화

`governance_lab` 스키마에서 **예측 최적화**를 활성화하여 향후 `VACUUM` 및 `OPTIMIZE` 유지 관리가 수동 스케줄링 없이 자동으로 실행되도록 하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks에서 Unity Catalog 스키마에 예측 최적화를 어떻게 활성화하나요?"*

**힌트:** 다음을 사용하세요: `ALTER SCHEMA ... ENABLE PREDICTIVE OPTIMIZATION`.

In [ ]:
%sql
ALTER SCHEMA automotive_catalog.governance_lab
ENABLE PREDICTIVE OPTIMIZATION;

## 연습 4: 데이터 계보를 프로그래밍 방식으로 쿼리

> 📋 **이 연습을 실행하기 전에**: 아직 하지 않았다면 여기서 잠시 멈추고 랩 설정 안내의 **Catalog Explorer 계보 단계**를 따라 UI에서 계보를 시각적으로 살펴보세요. 그런 다음 여기로 돌아와 SQL을 사용하여 계보 데이터를 쿼리하세요.

Unity Catalog는 모든 읽기 및 쓰기 이벤트를 캡처하여 `system.access.table_lineage` 시스템 테이블에 저장합니다. 이를 통해 다음과 같은 질문에 프로그래밍 방식으로 답할 수 있습니다: *"이번 주 우리 automotive 카탈로그에서 가장 많이 쿼리된 테이블은 무엇인가요?"*

### 작업 4.1 — 카탈로그에서 최근 액세스한 테이블 찾기

`system.access.table_lineage`를 쿼리하여 지난 7일 이내에 `automotive_catalog`와 관련된 모든 **쓰기 이벤트**(소스 테이블)를 찾으세요. 테이블 이름과 고유 이벤트 수를 반환하되, 가장 활발한 테이블 순으로 정렬하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks에서 최근 액세스한 Unity Catalog 테이블을 찾기 위해 system.access.table_lineage를 어떻게 쿼리하나요?"*

**힌트:** `source_table_full_name LIKE 'automotive_catalog%'`로 필터링하고 테이블 이름별로 그룹화하세요.

In [ ]:
%sql
SELECT
  source_table_full_name,
  COUNT(DISTINCT event_id) AS event_count
FROM system.access.table_lineage
WHERE
  event_date > CURRENT_DATE() - INTERVAL 7 DAYS
  AND source_table_full_name LIKE 'automotive_catalog%'
GROUP BY source_table_full_name
ORDER BY event_count DESC;

### 작업 4.2 — vehicle_telemetry의 기록 보기

`DESCRIBE HISTORY`를 사용하여 `vehicle_telemetry`의 전체 Delta Lake 작업 기록을 확인하세요. 삭제가 발생한 버전(`DELETE` 작업)을 식별하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Databricks SQL에서 Delta Lake 테이블의 전체 버전 기록을 어떻게 보나요?"*

In [ ]:
%sql
DESCRIBE HISTORY automotive_catalog.governance_lab.vehicle_telemetry;

## 연습 5: 감사 로깅

AutoSphere의 규정 준수 팀은 지난 30일 동안 권한이 있는 사용자만 고객 데이터에 액세스했음을 규제 기관에 증명해야 합니다. 또한 이 팀은 Unity Catalog 메타스토어에서 권한이 수정될 때마다 알림을 받기를 원합니다.

이 두 가지 요구 사항을 모두 충족하기 위해 `system.access.audit` 시스템 테이블을 사용합니다.

### 작업 5.1 — customer_registrations 테이블에 액세스한 사용자 찾기

`system.access.audit`를 쿼리하여 지난 30일 동안 `automotive_catalog.governance_lab.customer_registrations`에 액세스한 모든 사용자를 나열하세요. 각 액세스의 작업 이름과 시간을 포함하되, 최신 순으로 정렬하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"특정 Unity Catalog 테이블에 누가 액세스했는지 찾기 위해 Databricks 감사 로그 시스템 테이블을 어떻게 쿼리하나요?"*

**힌트:** `request_params.full_name_arg`와 `action_name IN ('getTable', 'createTable')`로 필터링하세요. 사용자 열에는 `user_identity.email`을 사용하세요.

In [ ]:
%sql
SELECT
  user_identity.email  AS user_email,
  action_name          AS access_type,
  event_time           AS access_time
FROM system.access.audit
WHERE
  request_params.full_name_arg = 'automotive_catalog.governance_lab.customer_registrations'
  AND action_name IN ('getTable', 'createTable', 'deleteTable')
  AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC;

### 작업 5.2 — 카탈로그 전반의 권한 변경 감지

`system.access.audit`를 쿼리하여 지난 30일 동안 `automotive_catalog`의 보안 가능 객체에 대한 모든 **권한 변경 이벤트**(`updatePermissions` 작업)를 나열하세요. 누가 변경했는지와 어떤 객체가 영향을 받았는지 포함하세요.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"특정 Unity Catalog 카탈로그에 대한 Databricks 감사 로그에서 모든 권한 변경을 어떻게 찾나요?"*

**힌트:** `service_name = 'unityCatalog'`, `action_name = 'updatePermissions'`, 그리고 `request_params.securable_full_name LIKE 'automotive_catalog%'`로 필터링하세요.

In [ ]:
%sql
SELECT
  event_time,
  user_identity.email                      AS changed_by,
  request_params.securable_type            AS object_type,
  request_params.securable_full_name       AS object_name,
  request_params.changes                   AS permission_changes
FROM system.access.audit
WHERE
  service_name = 'unityCatalog'
  AND action_name = 'updatePermissions'
  AND request_params.securable_full_name LIKE 'automotive_catalog%'
  AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC;

### 작업 5.3 — 사용자별 감사 활동 요약

지난 7일 동안 Unity Catalog 작업에 대해 사용자별 감사 이벤트의 총 개수를 세는 요약 쿼리를 만드세요. 이를 통해 규정 준수 팀은 플랫폼에서 누가 가장 활발했는지 빠르게 파악할 수 있습니다.

> 🤖 **Genie Code 팁:** 다음과 같이 물어보세요:
> *"Unity Catalog 활동에 대해 사용자별 이벤트를 세기 위해 Databricks 감사 로그를 어떻게 집계하나요?"*

In [ ]:
%sql
SELECT
  user_identity.email  AS user_email,
  COUNT(*)             AS total_events
FROM system.access.audit
WHERE
  service_name = 'unityCatalog'
  AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY user_identity.email
ORDER BY total_events DESC;